# P1-E16.7D-3 renamed(P1-E11.D_3) Representation-Controlled and Classifier-Controlled Fairness Audit

## Purpose

This development-only audit concerns that performance differences may reflect unequal inputs rather than classifier quality. It separates two questions:

1. **Classifier-controlled block:** logistic regression, linear support-vector machine, random forest, and a shallow multilayer perceptron receive the identical locked 646-feature vector for every recording.
2. **Time-series-input-controlled block:** CNN–BiLSTM, CEEDNet, and self-supervised multiresolution attention use the identical preprocessed 19-channel recording tensors, maximum epoch count, outer folds, metric implementation, and candidate budget already authenticated in P1-E16.7B.

The functional-connectivity GNN is excluded from the identical-time-series block because connectivity construction is part of its method-specific representation. It remains available in the complete-method comparison but cannot support an identical-feature claim.

Only development out-of-fold evidence is used. No official-test or external-data prediction is loaded, generated, or reconsidered.

## 1. Configuration and environment


In [ ]:
from pathlib import Path
import hashlib, json, platform, sys, time, warnings
from datetime import datetime, timezone
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, accuracy_score, f1_score, matthews_corrcoef, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

SEED=20261010
INNER_FOLDS=4
N_BOOT=3000
N_JOBS=-1
TASKS={'normal_vs_ad':['Normal','AD'],'mci_vs_ad':['MCI','AD'],
       'normal_vs_mci':['Normal','MCI'],'three_class':['Normal','MCI','AD']}
FEATURE_METHODS=['Logistic regression','Linear SVM','Random forest','Shallow MLP']
TIME_METHODS=['CNN–BiLSTM feature fusion','CEEDNet adaptation','SSL multiresolution attention']

PROJECT_ROOT=Path(r'D:\EEG_AD_Research\dementia\paper one')
OUTPUTS_ROOT=PROJECT_ROOT/'paper1_outputs'
E16_7A_ROOT=OUTPUTS_ROOT/'p1_e16_7a_four_endpoint_nested_conventional_freeze'
E16_7B_ROOT=OUTPUTS_ROOT/'p1_e16_7b_r_unified_four_endpoint_nested_method_comparison'
OUTPUT_ROOT=OUTPUTS_ROOT/'p1_e16_7d_3_representation_classifier_fairness_audit'
TABLES=OUTPUT_ROOT/'tables'; FIGURES=OUTPUT_ROOT/'figures'; CHECKPOINTS=OUTPUT_ROOT/'checkpoints'
for p in (TABLES,FIGURES,CHECKPOINTS): p.mkdir(parents=True,exist_ok=True)

INPUT_MANIFEST=E16_7A_ROOT/'tables'/'01_INPUT_PROVENANCE_MANIFEST.csv'
OUTER_FOLDS_FILE=E16_7A_ROOT/'tables'/'07_OUTER_SPLIT_AUDIT.csv'
E16_7A_INTEGRITY=E16_7A_ROOT/'tables'/'15_FINAL_INTEGRITY_CHECKS.csv'
TIME_PREDICTIONS=E16_7B_ROOT/'tables'/'08_ALL_METHOD_NESTED_OOF_PREDICTIONS.csv'
E16_7B_INTEGRITY=E16_7B_ROOT/'tables'/'12_FINAL_INTEGRITY_CHECKS.csv'

def sha256_file(path,chunk=1<<20):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()
def save_table(frame,name):
    path=TABLES/f'{name}.csv'; frame.to_csv(path,index=False); return path
def json_default(x):
    if isinstance(x,(np.integer,)): return int(x)
    if isinstance(x,(np.floating,)): return float(x)
    if isinstance(x,np.ndarray): return x.tolist()
    if isinstance(x,Path): return str(x)
    raise TypeError(type(x).__name__)
def holm_adjust(pvalues):
    p=np.asarray(pvalues,float); order=np.argsort(p); out=np.empty(len(p),float); running=0.0
    for rank,idx in enumerate(order): running=max(running,(len(p)-rank)*p[idx]); out[idx]=min(1.0,running)
    return out
print({'python':sys.version.split()[0],'platform':platform.platform(),'output_root':str(OUTPUT_ROOT)})

## 2. Authenticate and load the common development evidence


In [ ]:
required=[INPUT_MANIFEST,OUTER_FOLDS_FILE,E16_7A_INTEGRITY,TIME_PREDICTIONS,E16_7B_INTEGRITY]
missing=[str(p) for p in required if not p.is_file()]
if missing: raise FileNotFoundError('Missing upstream artifact(s):\n'+'\n'.join(missing))
for p in [E16_7A_INTEGRITY,E16_7B_INTEGRITY]:
    q=pd.read_csv(p)
    if not q['pass'].astype(str).str.lower().eq('true').all(): raise AssertionError(f'Upstream integrity failed: {p}')

source_manifest=pd.read_csv(INPUT_MANIFEST)
role_to_row={r.role:r for r in source_manifest.itertuples()}
if 'recording-level feature matrix' not in role_to_row or 'official partition manifest' not in role_to_row:
    raise KeyError('P1-E16.7A input manifest lacks the two required roles.')
feature_path=Path(role_to_row['recording-level feature matrix'].path)
partition_path=Path(role_to_row['official partition manifest'].path)
for role,path in [('feature matrix',feature_path),('partition manifest',partition_path)]:
    if not path.is_file(): raise FileNotFoundError(f'{role}: {path}')
expected_feature_hash=role_to_row['recording-level feature matrix'].sha256
expected_partition_hash=role_to_row['official partition manifest'].sha256
if sha256_file(feature_path)!=expected_feature_hash or sha256_file(partition_path)!=expected_partition_hash:
    raise AssertionError('A locked P1-E16.7A source hash changed.')

inputs=pd.DataFrame([
    {'role':'locked 646-feature matrix','path':str(feature_path),'sha256':sha256_file(feature_path)},
    {'role':'locked cohort/partition manifest','path':str(partition_path),'sha256':sha256_file(partition_path)},
    {'role':'canonical outer-fold audit','path':str(OUTER_FOLDS_FILE),'sha256':sha256_file(OUTER_FOLDS_FILE)},
    {'role':'time-series nested OOF predictions','path':str(TIME_PREDICTIONS),'sha256':sha256_file(TIME_PREDICTIONS)},])
save_table(inputs,'01_AUTHENTICATED_INPUTS'); display(inputs)

## 3. Align the identical 646-feature matrix to the canonical folds


In [ ]:
def first_key(keys,candidates):
    lookup={str(k).lower():k for k in keys}
    return next((lookup[c.lower()] for c in candidates if c.lower() in lookup),None)
with np.load(feature_path,allow_pickle=False) as z:
    keys=list(z.files); xk=first_key(keys,['X','features','feature_matrix']); ik=first_key(keys,['recording_id','recording_ids','series_id','ids']); nk=first_key(keys,['feature_names','features_names','columns'])
    if not all([xk,ik,nk]): raise KeyError(f'Feature archive lacks required arrays: {keys}')
    X=np.asarray(z[xk],np.float32); recording_ids=np.asarray(z[ik]).astype(str); feature_names=np.asarray(z[nk]).astype(str)
recording_ids=np.asarray([str(x).strip().zfill(5) for x in recording_ids])
if X.shape!=(len(recording_ids),len(feature_names)): raise AssertionError('Feature matrix shape mismatch.')
if len(feature_names)!=646: raise AssertionError(f'Expected the locked 646-feature representation, found {len(feature_names)}.')
if not np.isfinite(X).all(): raise AssertionError('Non-finite locked feature values detected.')

manifest=pd.read_csv(partition_path,dtype=str); manifest['recording_id']=manifest.recording_id.astype(str).str.strip().str.zfill(5)
labels=dict(zip(manifest.recording_id,manifest.experimental_label)); splits=dict(zip(manifest.recording_id,manifest.official_split.str.lower()))
feature_lookup={rid:i for i,rid in enumerate(recording_ids)}
folds=pd.read_csv(OUTER_FOLDS_FILE,dtype={'recording_id':str}).rename(columns={'endpoint':'task','fold':'outer_fold'})
folds['recording_id']=folds.recording_id.astype(str).str.strip().str.zfill(5)
if 'role' in folds: folds=folds.loc[folds.role.astype(str).str.lower().eq('outer_holdout')]
folds=folds.loc[folds.task.isin(TASKS),['task','outer_fold','recording_id']].drop_duplicates(['task','recording_id'])
folds['outer_fold']=folds.outer_fold.astype(int)
if not set(folds.recording_id).issubset(feature_lookup): raise AssertionError('Canonical fold IDs missing from feature matrix.')
if any(str(splits[r]).lower()=='test' for r in folds.recording_id): raise AssertionError('Official test entered the development fairness audit.')

scope=[]
for task,classes in TASKS.items():
    q=folds.loc[folds.task.eq(task)]; observed=[labels[r] for r in q.recording_id]
    scope.append({'task':task,'n':len(q),'n_features':X.shape[1],'five_outer_folds':set(q.outer_fold)==set(range(1,6)),**{f'n_{c}':observed.count(c) for c in classes}})
scope=pd.DataFrame(scope); save_table(scope,'02_FAIRNESS_COHORT_SCOPE'); display(scope)
if not scope.five_outer_folds.all(): raise AssertionError('Canonical folds incomplete.')

## 4. Classifier-controlled nested comparison on identical engineered features

Every classifier receives all 646 features in the same order. Each family has exactly two prespecified hyperparameter configurations. Imputation and scaling are fitted only inside the relevant training partition. Outer-fold memberships are inherited unchanged from P1-E16.7A.

In [ ]:
def classifier_spaces():
    return {
      'Logistic regression':(Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler()),('model',LogisticRegression(max_iter=3000,class_weight='balanced',random_state=SEED))]),{'model__C':[.01,.1]}),
      'Linear SVM':(Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler()),('model',CalibratedClassifierCV(LinearSVC(class_weight='balanced',random_state=SEED,dual='auto'),cv=3))]),{'model__estimator__C':[.01,.1]}),
      'Random forest':(Pipeline([('impute',SimpleImputer(strategy='median')),('model',RandomForestClassifier(n_estimators=400,class_weight='balanced_subsample',max_features='sqrt',min_samples_leaf=3,n_jobs=1,random_state=SEED))]),{'model__max_depth':[None,12]}),
      'Shallow MLP':(Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler()),('model',MLPClassifier(max_iter=500,early_stopping=True,validation_fraction=.15,alpha=1e-3,random_state=SEED))]),{'model__hidden_layer_sizes':[(64,),(128,64)]}),
    }
def aligned_probability(estimator,x,classes):
    raw=np.asarray(estimator.predict_proba(x),float); out=np.zeros((len(x),len(classes))); mapping={str(c):i for i,c in enumerate(estimator.classes_)}
    for j,c in enumerate(classes): out[:,j]=raw[:,mapping[str(c)]]
    return out/out.sum(1,keepdims=True)

prediction_rows=[]; selection_rows=[]
for task,classes in TASKS.items():
    qfold=folds.loc[folds.task.eq(task)]; endpoint_ids=qfold.recording_id.tolist(); endpoint_index=np.asarray([feature_lookup[r] for r in endpoint_ids]); endpoint_y=np.asarray([labels[r] for r in endpoint_ids])
    class_to_index={c:i for i,c in enumerate(classes)}; endpoint_y_index=np.asarray([class_to_index[y] for y in endpoint_y],dtype=int)
    for outer_fold in range(1,6):
        hold_ids=set(qfold.loc[qfold.outer_fold.eq(outer_fold),'recording_id']); test_mask=np.asarray([r in hold_ids for r in endpoint_ids]); train_mask=~test_mask
        xtr,xt=X[endpoint_index[train_mask]],X[endpoint_index[test_mask]]; ytr,yt=endpoint_y_index[train_mask],endpoint_y_index[test_mask]; test_ids=np.asarray(endpoint_ids)[test_mask]
        inner=StratifiedKFold(n_splits=INNER_FOLDS,shuffle=True,random_state=SEED+outer_fold)
        for method,(estimator,grid) in classifier_spaces().items():
            ckpt=CHECKPOINTS/f'{task}__fold{outer_fold}__{method.replace(" ","_")}.csv'; meta=ckpt.with_suffix('.json')
            if ckpt.is_file() and meta.is_file():
                cached=pd.read_csv(ckpt,dtype={'recording_id':str}); cached.recording_id=cached.recording_id.astype(str).str.zfill(5)
                if set(cached.recording_id)==set(test_ids) and not cached.recording_id.duplicated().any():
                    prediction_rows.extend(cached.to_dict('records')); selection_rows.append(json.loads(meta.read_text())); print('restored',task,outer_fold,method); continue
                raise RuntimeError(f'Invalid checkpoint: {ckpt}')
            # Integer targets avoid the scikit-learn MLP early-stopping bug that
            # applies np.isnan to string-valued validation predictions.
            jobs=1 if method=='Shallow MLP' else N_JOBS
            started=time.perf_counter(); search=GridSearchCV(estimator,grid,scoring='balanced_accuracy',cv=inner,n_jobs=jobs,refit=True,error_score='raise')
            search.fit(xtr,ytr); prob=aligned_probability(search.best_estimator_,xt,list(range(len(classes)))); predicted=np.asarray(classes)[prob.argmax(1)]
            local=[]
            for i,rid in enumerate(test_ids):
                row={'task':task,'outer_fold':outer_fold,'recording_id':rid,'true_label':classes[int(yt[i])],'predicted_label':predicted[i],'method':method,'representation':'identical_646_features'}
                row.update({f'probability_{c}':float(prob[i,j]) for j,c in enumerate(classes)}); local.append(row)
            pd.DataFrame(local).to_csv(ckpt,index=False); prediction_rows.extend(local)
            selected={'task':task,'outer_fold':outer_fold,'method':method,'best_params_json':json.dumps(search.best_params_,sort_keys=True),'inner_balanced_accuracy':float(search.best_score_),'seconds':time.perf_counter()-started}
            meta.write_text(json.dumps(selected,indent=2)); selection_rows.append(selected); print('complete',task,outer_fold,method)

feature_predictions=pd.DataFrame(prediction_rows); feature_selection=pd.DataFrame(selection_rows)
save_table(feature_predictions,'03_IDENTICAL_FEATURE_NESTED_OOF_PREDICTIONS'); save_table(feature_selection,'04_IDENTICAL_FEATURE_INNER_SELECTION_AUDIT')
display(feature_selection)

## 5. Reuse the authenticated identical-time-series comparison


In [ ]:
time_predictions=pd.read_csv(TIME_PREDICTIONS,dtype={'recording_id':str}).rename(columns={'endpoint':'task','fold':'outer_fold'})
time_predictions.recording_id=time_predictions.recording_id.astype(str).str.strip().str.zfill(5)
time_predictions=time_predictions.loc[time_predictions.method.isin(TIME_METHODS)&time_predictions.task.isin(TASKS)].copy()
time_predictions['representation']='identical_preprocessed_19_channel_epoch_tensor'
expected={(t,m) for t in TASKS for m in TIME_METHODS}; observed=set(map(tuple,time_predictions[['task','method']].drop_duplicates().to_numpy()))
if observed!=expected: raise AssertionError(f'Incomplete time-series method block: {expected-observed}')
for task,classes in TASKS.items():
    expected_ids=set(folds.loc[folds.task.eq(task),'recording_id'])
    for method in TIME_METHODS:
        q=time_predictions.loc[(time_predictions.task.eq(task))&time_predictions.method.eq(method)]
        if set(q.recording_id)!=expected_ids or q.recording_id.duplicated().any(): raise AssertionError(f'Time-series alignment failure: {task}, {method}')
save_table(time_predictions,'05_IDENTICAL_TIME_SERIES_NESTED_OOF_PREDICTIONS')
display(time_predictions.groupby(['task','method']).size().rename('n').reset_index())

## 6. Common metrics and within-block paired comparisons


In [ ]:
def metrics_for(q,task):
    classes=TASKS[task]; c2i={c:i for i,c in enumerate(classes)}; y=q.true_label.map(c2i).to_numpy(int); yp=q.predicted_label.map(c2i).to_numpy(int); p=q[[f'probability_{c}' for c in classes]].to_numpy(float)
    return {'n':len(q),'balanced_accuracy':balanced_accuracy_score(y,yp),'accuracy':accuracy_score(y,yp),'macro_f1':f1_score(y,yp,average='macro'),'mcc':matthews_corrcoef(y,yp),'roc_auc':roc_auc_score(y,p[:,1]) if len(classes)==2 else roc_auc_score(y,p,multi_class='ovr',average='macro',labels=np.arange(len(classes)))}
blocks={'Identical engineered features':feature_predictions,'Identical EEG time-series input':time_predictions}
metric_rows=[]
for block,frame in blocks.items():
    for (task,method),q in frame.groupby(['task','method']): metric_rows.append({'comparison_block':block,'task':task,'method':method,**metrics_for(q,task)})
metrics=pd.DataFrame(metric_rows); save_table(metrics,'06_CONTROLLED_BLOCK_PERFORMANCE'); display(metrics.sort_values(['comparison_block','task','balanced_accuracy'],ascending=[True,True,False]).round(4))

def paired_bootstrap(frame,task,method,reference,rng):
    a=frame.loc[(frame.task.eq(task))&frame.method.eq(method)].set_index('recording_id').sort_index(); b=frame.loc[(frame.task.eq(task))&frame.method.eq(reference)].set_index('recording_id').sort_index()
    if not a.index.equals(b.index) or not a.true_label.equals(b.true_label): raise AssertionError('Paired alignment failure.')
    classes=TASKS[task]; strata={c:np.flatnonzero(a.true_label.to_numpy()==c) for c in classes}; delta=[]
    for _ in range(N_BOOT):
        ix=np.concatenate([rng.choice(strata[c],len(strata[c]),replace=True) for c in classes]); delta.append(metrics_for(a.iloc[ix],task)['balanced_accuracy']-metrics_for(b.iloc[ix],task)['balanced_accuracy'])
    delta=np.asarray(delta); observed=metrics_for(a,task)['balanced_accuracy']-metrics_for(b,task)['balanced_accuracy']; p=2*min((np.sum(delta<=0)+1)/(N_BOOT+1),(np.sum(delta>=0)+1)/(N_BOOT+1))
    return observed,np.quantile(delta,.025),np.quantile(delta,.975),min(1,p)

rng=np.random.default_rng(SEED); comparisons=[]
for block,frame,reference,methods in [
    ('Identical engineered features',feature_predictions,'Logistic regression',FEATURE_METHODS),
    ('Identical EEG time-series input',time_predictions,'CNN–BiLSTM feature fusion',TIME_METHODS)]:
    for task in TASKS:
        start=len(comparisons)
        for method in methods:
            if method==reference: continue
            d,lo,hi,p=paired_bootstrap(frame,task,method,reference,rng); comparisons.append({'comparison_block':block,'task':task,'method':method,'reference':reference,'paired_ba_difference':d,'ci_low':lo,'ci_high':hi,'unadjusted_p_value':p})
        adjusted=holm_adjust([r['unadjusted_p_value'] for r in comparisons[start:]])
        for row,padj in zip(comparisons[start:],adjusted): row['holm_adjusted_p_value']=padj
comparisons=pd.DataFrame(comparisons); save_table(comparisons,'07_WITHIN_BLOCK_PAIRED_COMPARISONS'); display(comparisons.round(4))

## 7. Publication-ready controlled-comparison figures


In [ ]:
endpoint_labels={'normal_vs_ad':'Normal vs AD','mci_vs_ad':'MCI vs AD','normal_vs_mci':'Normal vs MCI','three_class':'Three class'}
for block,filename in [('Identical engineered features','Figure_P1_E16_7D3_Identical_Feature_Comparison'),('Identical EEG time-series input','Figure_P1_E16_7D3_Identical_TimeSeries_Comparison')]:
    q=metrics.loc[metrics.comparison_block.eq(block)]; methods=FEATURE_METHODS if 'engineered' in block else TIME_METHODS
    fig,axes=plt.subplots(2,2,figsize=(13,9))
    for ax,task in zip(axes.flat,TASKS):
        z=q.loc[q.task.eq(task)].set_index('method').reindex(methods); ax.bar(np.arange(len(z)),z.balanced_accuracy,color='#4c78a8')
        ax.axhline(1/len(TASKS[task]),ls='--',color='#b22222'); ax.set_xticks(np.arange(len(z))); ax.set_xticklabels(z.index,rotation=25,ha='right'); ax.set_ylim(max(0,1/len(TASKS[task])-.12),1); ax.set_title(endpoint_labels[task]); ax.set_ylabel('Development OOF balanced accuracy'); ax.grid(axis='y',alpha=.25)
    fig.suptitle(block,fontweight='bold'); fig.tight_layout(); fig.savefig(FIGURES/f'{filename}.pdf',bbox_inches='tight'); fig.savefig(FIGURES/f'{filename}.png',dpi=400,bbox_inches='tight'); plt.show()

## 8. Final fairness and provenance integrity lock


In [ ]:
checks=[]
for task,classes in TASKS.items():
    expected=set(folds.loc[folds.task.eq(task),'recording_id'])
    for block,frame,methods in [('feature',feature_predictions,FEATURE_METHODS),('time_series',time_predictions,TIME_METHODS)]:
        for method in methods:
            q=frame.loc[(frame.task.eq(task))&frame.method.eq(method)]; p=q[[f'probability_{c}' for c in classes]].to_numpy(float)
            checks.extend([
                {'scope':task,'check':f'{block}/{method}: identical recording membership','pass':set(q.recording_id)==expected and not q.recording_id.duplicated().any()},
                {'scope':task,'check':f'{block}/{method}: canonical folds','pass':set(q.outer_fold.astype(int))==set(range(1,6))},
                {'scope':task,'check':f'{block}/{method}: probabilities valid','pass':np.isfinite(p).all() and np.allclose(p.sum(1),1,atol=1e-6)},])
checks.extend([
    {'scope':'all','check':'feature block uses exactly 646 identical ordered features','pass':X.shape[1]==646},
    {'scope':'all','check':'feature-family candidate budgets equal two','pass':all(len(grid[next(iter(grid))])==2 for _,grid in classifier_spaces().values())},
    {'scope':'all','check':'time-series block excludes connectivity GNN','pass':'Functional-connectivity GNN' not in set(time_predictions.method)},
    {'scope':'all','check':'official test excluded','pass':not any(str(splits[r]).lower()=='test' for r in folds.recording_id)},
    {'scope':'all','check':'external datasets excluded','pass':True},
    {'scope':'all','check':'no performance target used','pass':True},
    {'scope':'all','check':'paired comparisons complete','pass':len(comparisons)==20 and comparisons.holm_adjusted_p_value.notna().all()},])
checks=pd.DataFrame(checks); save_table(checks,'08_FINAL_INTEGRITY_CHECKS'); display(checks)
if not checks['pass'].all(): raise AssertionError('P1-E16.7D-3 fairness integrity failure.')

run={'experiment':'P1-E16.7D-3 representation-controlled and classifier-controlled fairness audit','status':'PASSED','completed_utc':datetime.now(timezone.utc).isoformat(),
     'endpoints':TASKS,'feature_block_methods':FEATURE_METHODS,'time_series_block_methods':TIME_METHODS,'feature_count':int(X.shape[1]),
     'official_test_accessed':False,'external_data_accessed':False,'performance_target_used':False,'metrics':metrics.to_dict('records'),
     'comparisons':comparisons.to_dict('records'),'integrity_passed':True,'input_files':inputs.to_dict('records')}
(OUTPUT_ROOT/'run_record.json').write_text(json.dumps(run,indent=2,default=json_default),encoding='utf-8')
print('P1-E16.7D-3 PASSED — REPRESENTATION- AND CLASSIFIER-CONTROLLED FAIRNESS AUDIT COMPLETE')